# Audio Adversarial Inspection Notebook

Inspect and compare untargeted adversarial examples from fine-tuned AudioMAE and BEATs runs.

This notebook supports:
- filtering by model, checkpoint label, layer, sample/example id, and attack config
- inline listening for original/adversarial/perturbation variants
- spectrogram and waveform comparisons for a selected example
- summary plots across selected runs
- CSV export of the selected subset

## Expected artifact format

The notebook expects adversarial outputs under:

```text
experiments/audio/<model_name>/adversarial/<attack_id>/<layer_name>/
  metadata.jsonl
  summary.csv
  idxXXXX_<label>_original.pt
  idxXXXX_<label>_adversarial.pt
  idxXXXX_<label>_delta.pt
  idxXXXX_<label>_original.wav
  idxXXXX_<label>_adversarial.wav
  idxXXXX_<label>_perturbation.wav
```

where each `metadata.jsonl` record includes nested `attack_config`, `attack_trace`, and `saved_paths` fields.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, Markdown, display

plt.rcParams["figure.figsize"] = (10, 4)

DEFAULT_MODELS = ["audiomae_as2m_ft_as20k", "beats_iter3_plus_as2m"]
DEFAULT_EXP_ROOT = Path("../experiments/audio")
AB_GAP_SECONDS = 0.15
EXPORT_DEFAULT_PATH = Path("outputs/audio_adversarial_selected_summary.csv")

In [ ]:
def _workspace_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "experiments").exists():
        return cwd
    if (cwd.parent / "experiments").exists():
        return cwd.parent
    return cwd


WORKSPACE_ROOT = _workspace_root()
EXP_ROOT = (WORKSPACE_ROOT / DEFAULT_EXP_ROOT).resolve() if not DEFAULT_EXP_ROOT.is_absolute() else DEFAULT_EXP_ROOT


def parse_index_spec(spec: str) -> list[int]:
    spec = spec.strip()
    if not spec:
        return []
    parsed: list[int] = []
    for part in spec.split(","):
        token = part.strip()
        if not token:
            continue
        if "-" in token:
            lo_str, hi_str = token.split("-", 1)
            lo = int(lo_str)
            hi = int(hi_str)
            step = 1 if hi >= lo else -1
            parsed.extend(range(lo, hi + step, step))
        else:
            parsed.append(int(token))
    return sorted(set(parsed))


def parse_attack_id(attack_id: str) -> dict[str, Any]:
    parsed: dict[str, Any] = {
        "attack_mode_from_id": None,
        "norm_from_id": None,
        "epsilon_from_id": np.nan,
        "num_steps_from_id": np.nan,
        "loss_type_from_id": None,
    }
    if not attack_id:
        return parsed
    match = re.match(r"^(?P<mode>[^_]+)_(?P<norm>[^_]+)_eps_(?P<eps>[^_]+)_steps_(?P<steps>[^_]+)_loss_(?P<loss>.+)$", attack_id)
    if not match:
        return parsed
    parsed["attack_mode_from_id"] = match.group("mode")
    parsed["norm_from_id"] = match.group("norm")
    parsed["epsilon_from_id"] = float(match.group("eps").replace("p", "."))
    parsed["num_steps_from_id"] = int(match.group("steps"))
    parsed["loss_type_from_id"] = match.group("loss")
    return parsed


def resolve_artifact_path(raw_path: str | None, metadata_path: Path) -> str | None:
    if not raw_path:
        return None
    p = Path(raw_path)
    if p.is_absolute():
        return str(p)
    if p.exists():
        return str(p.resolve())
    from_workspace = (WORKSPACE_ROOT / p).resolve()
    if from_workspace.exists():
        return str(from_workspace)
    from_meta = (metadata_path.parent / p).resolve()
    return str(from_meta)


def derive_checkpoint_label(record: dict[str, Any], model_name: str, attack_id: str, run_dir: str) -> str:
    for key in ("checkpoint_label", "checkpoint_name", "checkpoint", "checkpoint_path"):
        val = record.get(key)
        if isinstance(val, str) and val.strip():
            return Path(val).stem
    attack_cfg = record.get("attack_config", {})
    ckpt_in_cfg = attack_cfg.get("checkpoint") if isinstance(attack_cfg, dict) else None
    if isinstance(ckpt_in_cfg, str) and ckpt_in_cfg.strip():
        return Path(ckpt_in_cfg).stem
    return f"{model_name}:{attack_id}:{Path(run_dir).name}"


def flatten_metadata_record(record: dict[str, Any], metadata_path: Path) -> dict[str, Any]:
    layer_name = metadata_path.parent.name
    attack_id = metadata_path.parent.parent.name
    model_name = metadata_path.parent.parent.parent.parent.name
    run_dir = str(metadata_path.parent)
    source_info = record.get("source_info", {}) or {}
    attack_config = record.get("attack_config", {}) or {}
    saved_paths = record.get("saved_paths", {}) or {}
    attack_trace = record.get("attack_trace", {}) or {}
    parsed_from_attack_id = parse_attack_id(attack_id)

    row = {
        "model_name": record.get("model_name", model_name),
        "attack_id": record.get("attack_id", attack_id),
        "layer_name": record.get("layer_name", layer_name),
        "sample_idx": record.get("sample_idx"),
        "class_label": record.get("class_label"),
        "source_ytid": source_info.get("ytid"),
        "source_labels": source_info.get("labels"),
        "sample_rate": record.get("sample_rate"),
        "run_dir": run_dir,
        "metadata_jsonl": str(metadata_path),
        "attack_mode": record.get("attack_mode") or parsed_from_attack_id["attack_mode_from_id"],
        "norm": attack_config.get("norm") or parsed_from_attack_id["norm_from_id"],
        "epsilon": attack_config.get("epsilon", parsed_from_attack_id["epsilon_from_id"]),
        "step_size": attack_config.get("step_size"),
        "num_steps": attack_config.get("num_steps", parsed_from_attack_id["num_steps_from_id"]),
        "num_random_starts": attack_config.get("num_random_starts"),
        "loss_type": attack_config.get("loss_type") or parsed_from_attack_id["loss_type_from_id"],
        "representation_distance_from_source": record.get("representation_distance_from_source"),
        "representation_distance_to_target": record.get("representation_distance_to_target"),
        "delta_l2": record.get("delta_l2"),
        "delta_linf": record.get("delta_linf"),
        "snr_db": record.get("snr_db"),
        "source_logits": record.get("source_logits"),
        "adversarial_logits": record.get("adversarial_logits"),
        "attack_trace": attack_trace,
        "saved_paths": saved_paths,
        "saved_paths_json": json.dumps(saved_paths, sort_keys=True),
    }

    for artifact_name in (
        "original_pt",
        "adversarial_pt",
        "delta_pt",
        "original_wav",
        "adversarial_wav",
        "delta_wav",
    ):
        row[artifact_name] = resolve_artifact_path(saved_paths.get(artifact_name), metadata_path)

    row["checkpoint_label"] = derive_checkpoint_label(
        record=record,
        model_name=row["model_name"],
        attack_id=row["attack_id"],
        run_dir=row["run_dir"],
    )
    row["clip_id"] = row["source_ytid"] if row.get("source_ytid") else row.get("sample_idx")
    return row


def read_jsonl_records(path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with path.open("r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def build_index(exp_root: Path = EXP_ROOT, model_allowlist: list[str] | None = None) -> pd.DataFrame:
    metadata_paths = sorted(exp_root.glob("*/adversarial/*/*/metadata.jsonl"))
    rows: list[dict[str, Any]] = []
    for metadata_path in metadata_paths:
        records = read_jsonl_records(metadata_path)
        for record in records:
            row = flatten_metadata_record(record, metadata_path)
            if model_allowlist and row["model_name"] not in model_allowlist:
                continue
            rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    numeric_cols = [
        "sample_idx",
        "sample_rate",
        "epsilon",
        "step_size",
        "num_steps",
        "num_random_starts",
        "representation_distance_from_source",
        "representation_distance_to_target",
        "delta_l2",
        "delta_linf",
        "snr_db",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values(["model_name", "attack_id", "layer_name", "sample_idx"]).reset_index(drop=True)
    return df

In [ ]:
all_df = build_index(EXP_ROOT, model_allowlist=DEFAULT_MODELS)
filtered_df = all_df.copy()

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Experiment root: {EXP_ROOT}")
print(f"Loaded rows: {len(all_df)}")

if all_df.empty:
    print("No adversarial metadata found. Confirm EXP_ROOT and run outputs.")
else:
    display(
        all_df.groupby(["model_name", "checkpoint_label", "layer_name"], dropna=False)
        .size()
        .rename("num_examples")
        .reset_index()
        .sort_values(["model_name", "layer_name"])
    )
    display(all_df.head(10))

In [ ]:
def options_from(df: pd.DataFrame, col: str) -> tuple[Any, ...]:
    if df.empty or col not in df.columns:
        return tuple()
    values = [v for v in sorted(df[col].dropna().unique().tolist())]
    return tuple(values)


model_widget = widgets.SelectMultiple(options=options_from(all_df, "model_name"), value=options_from(all_df, "model_name"), description="Model", rows=4)
checkpoint_widget = widgets.SelectMultiple(options=options_from(all_df, "checkpoint_label"), description="Checkpoint", rows=6)
layer_widget = widgets.SelectMultiple(options=options_from(all_df, "layer_name"), description="Layer", rows=8)
attack_id_widget = widgets.SelectMultiple(options=options_from(all_df, "attack_id"), description="Attack ID", rows=6)
epsilon_widget = widgets.SelectMultiple(options=options_from(all_df, "epsilon"), description="Epsilon", rows=4)
norm_widget = widgets.SelectMultiple(options=options_from(all_df, "norm"), description="Norm", rows=3)
loss_widget = widgets.SelectMultiple(options=options_from(all_df, "loss_type"), description="Loss", rows=4)
steps_widget = widgets.SelectMultiple(options=options_from(all_df, "num_steps"), description="Steps", rows=4)
sample_idx_text = widgets.Text(value="", description="Sample IDs", placeholder="e.g. 0-9,12")
clip_id_text = widgets.Text(value="", description="Clip ID", placeholder="substring match")
distance_threshold_widget = widgets.FloatText(value=0.05, description="Success thr")
apply_button = widgets.Button(description="Apply Filters", button_style="primary")
clear_button = widgets.Button(description="Clear", button_style="")
filter_output = widgets.Output()


def apply_filters(_: Any | None = None) -> None:
    global filtered_df
    df = all_df.copy()
    if model_widget.value:
        df = df[df["model_name"].isin(model_widget.value)]
    if checkpoint_widget.value:
        df = df[df["checkpoint_label"].isin(checkpoint_widget.value)]
    if layer_widget.value:
        df = df[df["layer_name"].isin(layer_widget.value)]
    if attack_id_widget.value:
        df = df[df["attack_id"].isin(attack_id_widget.value)]
    if epsilon_widget.value:
        df = df[df["epsilon"].isin(epsilon_widget.value)]
    if norm_widget.value:
        df = df[df["norm"].isin(norm_widget.value)]
    if loss_widget.value:
        df = df[df["loss_type"].isin(loss_widget.value)]
    if steps_widget.value:
        df = df[df["num_steps"].isin(steps_widget.value)]

    sample_spec = sample_idx_text.value.strip()
    if sample_spec:
        requested_indices = parse_index_spec(sample_spec)
        df = df[df["sample_idx"].isin(requested_indices)]

    clip_substring = clip_id_text.value.strip()
    if clip_substring:
        df = df[df["clip_id"].astype(str).str.contains(clip_substring, case=False, na=False)]

    filtered_df = df.sort_values(["model_name", "checkpoint_label", "layer_name", "sample_idx"]).reset_index(drop=True)
    if "refresh_example_options" in globals():
        refresh_example_options()

    with filter_output:
        filter_output.clear_output()
        print(f"Selected rows: {len(filtered_df)} / {len(all_df)}")
        cols = [
            "model_name",
            "checkpoint_label",
            "attack_id",
            "layer_name",
            "sample_idx",
            "clip_id",
            "epsilon",
            "num_steps",
            "representation_distance_from_source",
            "delta_l2",
            "snr_db",
        ]
        cols = [c for c in cols if c in filtered_df.columns]
        display(filtered_df[cols].head(30))


def clear_filters(_: Any | None = None) -> None:
    checkpoint_widget.value = tuple()
    layer_widget.value = tuple()
    attack_id_widget.value = tuple()
    epsilon_widget.value = tuple()
    norm_widget.value = tuple()
    loss_widget.value = tuple()
    steps_widget.value = tuple()
    sample_idx_text.value = ""
    clip_id_text.value = ""
    apply_filters()


apply_button.on_click(apply_filters)
clear_button.on_click(clear_filters)

display(
    widgets.VBox([
        widgets.HBox([model_widget, checkpoint_widget, layer_widget]),
        widgets.HBox([attack_id_widget, epsilon_widget, norm_widget, loss_widget, steps_widget]),
        widgets.HBox([sample_idx_text, clip_id_text, distance_threshold_widget]),
        widgets.HBox([apply_button, clear_button]),
        filter_output,
    ])
)

apply_filters()

In [ ]:
def load_waveform_from_pt(path: str | None) -> np.ndarray:
    if path is None:
        raise ValueError("Missing artifact path.")
    tensor = torch.load(path, map_location="cpu")
    if isinstance(tensor, dict):
        raise TypeError(f"Expected tensor artifact, got dict at {path}")
    waveform = torch.as_tensor(tensor).detach().cpu().float()
    if waveform.ndim == 3:
        waveform = waveform.squeeze(0)
    if waveform.ndim == 2:
        waveform = waveform.mean(dim=0)
    if waveform.ndim != 1:
        raise ValueError(f"Unexpected waveform shape at {path}: {tuple(waveform.shape)}")
    return waveform.numpy()


def log_spectrogram(x: np.ndarray, n_fft: int = 512, hop_length: int = 160, eps: float = 1e-8) -> np.ndarray:
    waveform = torch.as_tensor(x, dtype=torch.float32)
    stft = torch.stft(waveform, n_fft=n_fft, hop_length=hop_length, return_complex=True, center=True)
    spec = torch.log10(stft.abs().pow(2) + eps)
    return spec.numpy()


def normalize_for_listening(x: np.ndarray, peak: float = 0.95, eps: float = 1e-12) -> np.ndarray:
    max_abs = float(np.max(np.abs(x)))
    if max_abs < eps:
        return np.zeros_like(x)
    return (x / max_abs) * peak


def build_ab_loop(original: np.ndarray, adversarial: np.ndarray, sample_rate: int, loops: int = 2, gap_s: float = AB_GAP_SECONDS) -> np.ndarray:
    gap = np.zeros(int(sample_rate * gap_s), dtype=np.float32)
    chunks: list[np.ndarray] = []
    for _ in range(max(1, loops)):
        chunks.extend([original.astype(np.float32), gap, adversarial.astype(np.float32), gap])
    return np.concatenate(chunks)


def softmax_topk(logits: Any, k: int = 5) -> list[tuple[int, float]]:
    if logits is None:
        return []
    arr = np.asarray(logits, dtype=np.float32)
    arr = arr.reshape(-1)
    if arr.size == 0:
        return []
    exps = np.exp(arr - np.max(arr))
    probs = exps / np.sum(exps)
    top_idx = np.argsort(-probs)[: min(k, len(probs))]
    return [(int(i), float(probs[i])) for i in top_idx]


def plot_example_panels(row: pd.Series, original: np.ndarray, adversarial: np.ndarray, delta: np.ndarray, sr: int) -> None:
    spec_orig = log_spectrogram(original)
    spec_adv = log_spectrogram(adversarial)
    spec_delta = log_spectrogram(delta)
    spec_diff = spec_adv - spec_orig

    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    axes[0, 0].imshow(spec_orig, origin="lower", aspect="auto", cmap="magma")
    axes[0, 0].set_title("Original spectrogram")
    axes[0, 1].imshow(spec_adv, origin="lower", aspect="auto", cmap="magma")
    axes[0, 1].set_title("Adversarial spectrogram")
    axes[1, 0].imshow(spec_delta, origin="lower", aspect="auto", cmap="magma")
    axes[1, 0].set_title("Perturbation spectrogram")
    axes[1, 1].imshow(spec_diff, origin="lower", aspect="auto", cmap="coolwarm")
    axes[1, 1].set_title("Difference spectrogram (adv - orig)")
    for ax in axes.ravel():
        ax.set_xlabel("Frame")
        ax.set_ylabel("Freq bin")
    plt.show()

    t = np.arange(len(original), dtype=np.float32) / float(sr)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    axes[0].plot(t, original, label="original", alpha=0.9)
    axes[0].plot(t, adversarial, label="adversarial", alpha=0.75)
    axes[0].plot(t, delta, label="delta", alpha=0.7)
    axes[0].set_title("Waveform overlay")
    axes[0].set_xlabel("Time (s)")
    axes[0].legend(loc="upper right")

    steps = (((row.get("attack_trace") or {}).get("steps")) or [])
    if steps:
        step_df = pd.DataFrame(steps)
        axes[1].plot(step_df.get("step"), step_df.get("distance"), marker="o", ms=2)
        axes[1].set_title("Attack curve: distance over iterations")
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("Distance")
    else:
        axes[1].text(0.5, 0.5, "No attack trace steps found", ha="center", va="center")
        axes[1].set_axis_off()
    plt.show()


def render_metrics(row: pd.Series, threshold: float) -> None:
    dist = float(row.get("representation_distance_from_source", np.nan))
    success = bool(np.isfinite(dist) and dist > threshold)
    lines = [
        f"- model: `{row.get('model_name')}`",
        f"- checkpoint: `{row.get('checkpoint_label')}`",
        f"- layer: `{row.get('layer_name')}`",
        f"- sample_idx: `{row.get('sample_idx')}`",
        f"- clip_id: `{row.get('clip_id')}`",
        f"- attack_id: `{row.get('attack_id')}`",
        f"- epsilon: `{row.get('epsilon')}`",
        f"- steps: `{row.get('num_steps')}`",
        f"- representation_distance_from_source: `{row.get('representation_distance_from_source')}`",
        f"- delta_l2: `{row.get('delta_l2')}`",
        f"- delta_linf: `{row.get('delta_linf')}`",
        f"- SNR (dB): `{row.get('snr_db')}`",
        f"- attack_success (distance > {threshold}): `{success}`",
    ]
    display(Markdown("\n".join(lines)))

    source_topk = softmax_topk(row.get("source_logits"), k=5)
    adv_topk = softmax_topk(row.get("adversarial_logits"), k=5)
    if source_topk or adv_topk:
        display(Markdown("**Top-k predicted class indices/confidences (if logits available)**"))
        if source_topk:
            print("source top-k:", source_topk)
        if adv_topk:
            print("adversarial top-k:", adv_topk)

In [ ]:
example_dropdown = widgets.Dropdown(description="Example", options=[])
ab_checkbox = widgets.Checkbox(value=False, description="AB loop")
ab_loops_widget = widgets.IntSlider(value=2, min=1, max=8, step=1, description="AB loops")
render_button = widgets.Button(description="Render Example", button_style="primary")
example_output = widgets.Output()


def refresh_example_options() -> None:
    if filtered_df.empty:
        example_dropdown.options = []
        return
    options = []
    for i, row in filtered_df.iterrows():
        label = f"[{i}] {row['model_name']} | {row['checkpoint_label']} | {row['layer_name']} | sample={int(row['sample_idx'])}"
        options.append((label, i))
    example_dropdown.options = options
    example_dropdown.value = options[0][1] if options else None


def render_selected_example(_: Any | None = None) -> None:
    with example_output:
        example_output.clear_output()
        if filtered_df.empty or example_dropdown.value is None:
            print("No filtered examples. Apply filters first.")
            return
        row = filtered_df.loc[int(example_dropdown.value)]
        sr = int(row.get("sample_rate") or 16000)
        original = load_waveform_from_pt(row.get("original_pt"))
        adversarial = load_waveform_from_pt(row.get("adversarial_pt"))
        delta = load_waveform_from_pt(row.get("delta_pt"))
        delta_norm = normalize_for_listening(delta)

        render_metrics(row, threshold=float(distance_threshold_widget.value))

        display(Markdown("**Original audio**"))
        display(Audio(original, rate=sr))
        display(Markdown("**Adversarial audio**"))
        display(Audio(adversarial, rate=sr))
        display(Markdown("**Perturbation (true scale)**"))
        display(Audio(delta, rate=sr))
        display(Markdown("**Perturbation (normalized for listening)**"))
        display(Audio(delta_norm, rate=sr))

        if ab_checkbox.value:
            display(Markdown("**AB loop (original -> adversarial)**"))
            ab_audio = build_ab_loop(original, adversarial, sample_rate=sr, loops=int(ab_loops_widget.value))
            display(Audio(ab_audio, rate=sr))

        plot_example_panels(row, original, adversarial, delta, sr)


render_button.on_click(render_selected_example)
refresh_example_options()
display(widgets.VBox([widgets.HBox([example_dropdown, ab_checkbox, ab_loops_widget, render_button]), example_output]))

In [ ]:
def plot_summary(df: pd.DataFrame) -> None:
    if df.empty:
        print("No filtered data to summarize.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

    layer_groups = (
        df.dropna(subset=["representation_distance_from_source"])
        .groupby(["model_name", "layer_name"])["representation_distance_from_source"]
        .apply(list)
    )
    box_data = []
    box_labels = []
    for (model_name, layer_name), values in layer_groups.items():
        box_data.append(values)
        box_labels.append(f"{model_name}\n{layer_name}")
    if box_data:
        axes[0, 0].boxplot(box_data, labels=box_labels, showfliers=False)
        axes[0, 0].tick_params(axis="x", rotation=45)
    axes[0, 0].set_title("Representation distance by layer")
    axes[0, 0].set_ylabel("representation_distance_from_source")

    for model_name, g in df.groupby("model_name"):
        axes[0, 1].scatter(g["snr_db"], g["representation_distance_from_source"], alpha=0.7, label=model_name)
    axes[0, 1].set_title("SNR vs representation distance")
    axes[0, 1].set_xlabel("SNR (dB)")
    axes[0, 1].set_ylabel("representation_distance_from_source")
    axes[0, 1].legend(loc="best")

    for model_name, g in df.groupby("model_name"):
        axes[1, 0].scatter(g["delta_l2"], g["representation_distance_from_source"], alpha=0.7, label=model_name)
    axes[1, 0].set_title("Perturbation norm vs representation distance")
    axes[1, 0].set_xlabel("delta_l2")
    axes[1, 0].set_ylabel("representation_distance_from_source")
    axes[1, 0].legend(loc="best")

    model_summary = (
        df.groupby("model_name", dropna=False)
        .agg(
            num_examples=("sample_idx", "count"),
            mean_distance=("representation_distance_from_source", "mean"),
            mean_snr=("snr_db", "mean"),
            mean_delta_l2=("delta_l2", "mean"),
        )
        .reset_index()
    )
    x = np.arange(len(model_summary))
    width = 0.35
    axes[1, 1].bar(x - width / 2, model_summary["mean_distance"], width=width, label="mean_distance")
    axes[1, 1].bar(x + width / 2, model_summary["mean_snr"], width=width, label="mean_snr")
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(model_summary["model_name"], rotation=20)
    axes[1, 1].set_title("AudioMAE vs BEATs model comparison")
    axes[1, 1].legend(loc="best")
    plt.show()

    display(model_summary)


plot_summary(filtered_df)

In [ ]:
export_path_widget = widgets.Text(value=str(EXPORT_DEFAULT_PATH), description="Export CSV")
export_button = widgets.Button(description="Save Selected CSV", button_style="success")
export_output = widgets.Output()


def export_selected(_: Any | None = None) -> None:
    with export_output:
        export_output.clear_output()
        if filtered_df.empty:
            print("No selected rows to export.")
            return

        export_cols = [
            "model_name",
            "checkpoint_label",
            "attack_id",
            "layer_name",
            "sample_idx",
            "clip_id",
            "attack_mode",
            "norm",
            "epsilon",
            "step_size",
            "num_steps",
            "loss_type",
            "representation_distance_from_source",
            "delta_l2",
            "delta_linf",
            "snr_db",
            "original_pt",
            "adversarial_pt",
            "delta_pt",
            "original_wav",
            "adversarial_wav",
            "delta_wav",
            "metadata_jsonl",
            "run_dir",
            "saved_paths_json",
        ]
        keep_cols = [c for c in export_cols if c in filtered_df.columns]

        export_path = Path(export_path_widget.value)
        if not export_path.is_absolute():
            export_path = (Path.cwd() / export_path).resolve()
        export_path.parent.mkdir(parents=True, exist_ok=True)
        filtered_df[keep_cols].to_csv(export_path, index=False)
        print(f"Saved {len(filtered_df)} rows to: {export_path}")


export_button.on_click(export_selected)
display(widgets.VBox([widgets.HBox([export_path_widget, export_button]), export_output]))

## Validation checklist
- Run indexing and confirm rows are loaded for both AudioMAE and BEATs runs.
- Apply filters and click **Render Example** to verify players + plots.
- Run summary plotting cell for model/layer slices.
- Export selected rows and open the CSV to confirm artifact paths and metrics.